# Fetch Charity Water Images

This notebook fetches images from the Charity Water website, saves them into the repository, optimizes them, and prepares metadata for integration into the landing page.

In [1]:
# Install required packages
%pip install pillow beautifulsoup4 requests tqdm



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Section 1: Import required libraries
import os
import pathlib
import hashlib
import json
import subprocess
from urllib.parse import urljoin, urlparse
from pathlib import Path

import requests
from bs4 import BeautifulSoup
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display, HTML, Image as IPImage

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Section 2: Configure repo paths and VS Code context
root = Path('/workspaces/landing-page')
image_dir = root / 'public' / 'images' / 'charitywater'
image_dir.mkdir(parents=True, exist_ok=True)
metadata_path = image_dir / 'metadata.json'

print('Root directory:', root)
print('Image directory:', image_dir)
print('Existing image files:', sorted([p.name for p in image_dir.glob('*') if p.is_file()]))

Root directory: /workspaces/landing-page
Image directory: /workspaces/landing-page/public/images/charitywater
Existing image files: ['bbb-BCol5qPK.jpg', 'candid-DD7EBozV.jpg', 'charity-navigator-Caxpe_EG.jpg', 'charity-watch-BDZG6xjJ.jpg', 'metadata.json']


In [4]:
# Section 3: Scrape website for image URLs
base_url = 'https://www.charitywater.org'
response = requests.get(base_url, timeout=20)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')
img_urls = set()

for img in soup.find_all('img'):
    src = img.get('src') or img.get('data-src') or img.get('data-lazy-src')
    if src:
        img_urls.add(urljoin(base_url, src))
    srcset = img.get('srcset')
    if srcset:
        for item in srcset.split(','):
            url_part = item.strip().split(' ')[0]
            if url_part:
                img_urls.add(urljoin(base_url, url_part))

valid_ext = {'.jpg', '.jpeg', '.png', '.webp', '.svg'}
filtered_urls = []
for url in img_urls:
    parsed = urlparse(url)
    if not parsed.scheme.startswith('http'):
        continue
    ext = Path(parsed.path).suffix.lower()
    if ext in valid_ext:
        filtered_urls.append(url)

print('Found', len(filtered_urls), 'candidate image URLs')
filtered_urls[:20]

Found 4 candidate image URLs


['https://www.charitywater.org/vite/assets/candid-DD7EBozV.png',
 'https://www.charitywater.org/vite/assets/charity-watch-BDZG6xjJ.png',
 'https://www.charitywater.org/vite/assets/charity-navigator-Caxpe_EG.png',
 'https://www.charitywater.org/vite/assets/bbb-BCol5qPK.png']

In [5]:
# Section 4: Download and save images into repo (assets/images)

def safe_filename(url):
    parsed = urlparse(url)
    name = Path(parsed.path).name
    if not name:
        name = hashlib.sha1(url.encode('utf-8')).hexdigest() + '.jpg'
    return name

saved_images = []
for url in tqdm(filtered_urls, desc='Downloading images'):
    filename = safe_filename(url)
    target = image_dir / filename
    if target.exists():
        saved_images.append((url, target))
        continue
    try:
        with requests.get(url, stream=True, timeout=20) as r:
            r.raise_for_status()
            with open(target, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
        saved_images.append((url, target))
    except Exception as exc:
        print('Failed to download', url, exc)

print('Saved images:', len(saved_images))

Saved images: 4


In [6]:
# Section 5: Resize/optimize images with Pillow
optimized = []
for url, path in saved_images:
    try:
        with Image.open(path) as img:
            img = img.convert('RGB')
            width, height = img.size
            if width > 1600:
                ratio = 1600 / width
                img = img.resize((1600, int(height * ratio)), Image.LANCZOS)
            optimized_path = path.with_suffix('.jpg')
            img.save(optimized_path, format='JPEG', quality=85, optimize=True)
            if optimized_path != path:
                path.unlink()
            optimized.append((url, optimized_path))
    except Exception as exc:
        print('Optimize failed for', path, exc)

print('Optimized images:', len(optimized))

Optimized images: 4


/usr/local/python/3.12.1/lib/python3.12/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


In [7]:
# Section 6: Add alt text and metadata files
metadata = {}
for url, path in optimized:
    with Image.open(path) as img:
        metadata[path.name] = {
            'source_url': url,
            'filename': path.name,
            'width': img.width,
            'height': img.height,
            'alt': 'Charity Water campaign image',
            'license': 'See source website terms',
        }

with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print('Metadata written to', metadata_path)

Metadata written to /workspaces/landing-page/public/images/charitywater/metadata.json


In [8]:
# Section 7: Display images inside the notebook
html_images = []
for info in metadata.values():
    html_images.append(f"<div style='max-width: 320px; margin: 1rem;'><img src='public/images/charitywater/{info['filename']}' style='width:100%; border-radius:16px; box-shadow:0 12px 30px rgba(0,0,0,0.12);'><p><strong>{info['filename']}</strong><br><a href='{info['source_url']}' target='_blank'>Source</a></p></div>")

display(HTML('<div style="display:flex; flex-wrap:wrap; justify-content:center;">' + ''.join(html_images) + '</div>'))

In [9]:
# Section 8: Stage and commit saved images using Git
print('To add and commit the new images:')
print('  git add public/images/charitywater metadata.json')
print('  git commit -m "Add Charity Water website images and metadata"')
print('  git push')

To add and commit the new images:
  git add public/images/charitywater metadata.json
  git commit -m "Add Charity Water website images and metadata"
  git push


# Fetch Charity Water Images

This notebook fetches images from the Charity Water website, saves them into the repository, optimizes them, and prepares metadata for integration into the landing page.

In [10]:
# Section 1: Import required libraries
import os
import pathlib
import hashlib
import json
import subprocess
from urllib.parse import urljoin, urlparse
from pathlib import Path

import requests
from bs4 import BeautifulSoup
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display, HTML, Image as IPImage

In [11]:
# Section 2: Configure repo paths and VS Code context
root = Path('/workspaces/landing-page')
image_dir = root / 'public' / 'images' / 'charitywater'
image_dir.mkdir(parents=True, exist_ok=True)
metadata_path = image_dir / 'metadata.json'

print('Root directory:', root)
print('Image directory:', image_dir)
print('Existing image files:', sorted([p.name for p in image_dir.glob('*') if p.is_file()]))

Root directory: /workspaces/landing-page
Image directory: /workspaces/landing-page/public/images/charitywater
Existing image files: ['bbb-BCol5qPK.jpg', 'candid-DD7EBozV.jpg', 'charity-navigator-Caxpe_EG.jpg', 'charity-watch-BDZG6xjJ.jpg', 'metadata.json']


In [12]:
# Section 3: Scrape website for image URLs
base_url = 'https://www.charitywater.org'
response = requests.get(base_url, timeout=20)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')
img_urls = set()

for img in soup.find_all('img'):
    src = img.get('src') or img.get('data-src') or img.get('data-lazy-src')
    if src:
        img_urls.add(urljoin(base_url, src))
    srcset = img.get('srcset')
    if srcset:
        for item in srcset.split(','):
            url_part = item.strip().split(' ')[0]
            if url_part:
                img_urls.add(urljoin(base_url, url_part))

valid_ext = {'.jpg', '.jpeg', '.png', '.webp', '.svg'}
filtered_urls = []
for url in img_urls:
    parsed = urlparse(url)
    if not parsed.scheme.startswith('http'):
        continue
    ext = Path(parsed.path).suffix.lower()
    if ext in valid_ext:
        filtered_urls.append(url)

print('Found', len(filtered_urls), 'candidate image URLs')
filtered_urls[:20]

Found 4 candidate image URLs


['https://www.charitywater.org/vite/assets/candid-DD7EBozV.png',
 'https://www.charitywater.org/vite/assets/charity-watch-BDZG6xjJ.png',
 'https://www.charitywater.org/vite/assets/charity-navigator-Caxpe_EG.png',
 'https://www.charitywater.org/vite/assets/bbb-BCol5qPK.png']

In [13]:
# Section 4: Download and save images into repo (assets/images)

def safe_filename(url):
    parsed = urlparse(url)
    name = Path(parsed.path).name
    if not name:
        name = hashlib.sha1(url.encode('utf-8')).hexdigest() + '.jpg'
    return name

saved_images = []
for url in tqdm(filtered_urls, desc='Downloading images'):
    filename = safe_filename(url)
    target = image_dir / filename
    if target.exists():
        saved_images.append((url, target))
        continue
    try:
        with requests.get(url, stream=True, timeout=20) as r:
            r.raise_for_status()
            with open(target, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
        saved_images.append((url, target))
    except Exception as exc:
        print('Failed to download', url, exc)

print('Saved images:', len(saved_images))

Saved images: 4


In [14]:
# Section 5: Resize/optimize images with Pillow
optimized = []
for url, path in saved_images:
    try:
        with Image.open(path) as img:
            img = img.convert('RGB')
            width, height = img.size
            if width > 1600:
                ratio = 1600 / width
                img = img.resize((1600, int(height * ratio)), Image.LANCZOS)
            optimized_path = path.with_suffix('.jpg')
            img.save(optimized_path, format='JPEG', quality=85, optimize=True)
            if optimized_path != path:
                path.unlink()
            optimized.append((url, optimized_path))
    except Exception as exc:
        print('Optimize failed for', path, exc)

print('Optimized images:', len(optimized))

Optimized images: 4


In [15]:
# Section 6: Add alt text and metadata files
metadata = {}
for url, path in optimized:
    with Image.open(path) as img:
        metadata[path.name] = {
            'source_url': url,
            'filename': path.name,
            'width': img.width,
            'height': img.height,
            'alt': 'Charity Water campaign image',
            'license': 'See source website terms',
        }

with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print('Metadata written to', metadata_path)

Metadata written to /workspaces/landing-page/public/images/charitywater/metadata.json


In [16]:
# Section 7: Display images inside the notebook
html_images = []
for info in metadata.values():
    html_images.append(f"<div style='max-width: 320px; margin: 1rem;'><img src='public/images/charitywater/{info['filename']}' style='width:100%; border-radius:16px; box-shadow:0 12px 30px rgba(0,0,0,0.12);'><p><strong>{info['filename']}</strong><br><a href='{info['source_url']}' target='_blank'>Source</a></p></div>")

display(HTML('<div style="display:flex; flex-wrap:wrap; justify-content:center;">' + ''.join(html_images) + '</div>'))

In [17]:
# Section 8: Stage and commit saved images using Git
print('To add and commit the new images:')
print('  git add public/images/charitywater metadata.json')
print('  git commit -m "Add Charity Water website images and metadata"')
print('  git push')

To add and commit the new images:
  git add public/images/charitywater metadata.json
  git commit -m "Add Charity Water website images and metadata"
  git push
